# Advanced Few‑Shot Auto‑Selection + Prompt Chaining (LangChain vs “Normal” Python)

Goal: understand **why LangChain is helpful** for few-shot prompting:

1) **Normal technique**: you manually pick examples and format prompts.
2) **LangChain technique**: you plug in **ExampleSelectors** (semantic similarity / MMR) and chain them cleanly.

## What this notebook demonstrates
**Two advanced setups** (both run locally on an open model):

**Setup A — Dynamic Few‑Shot (Semantic Similarity)**
- Use embeddings + vector search to automatically pick the *most relevant* few-shot exemplars for each new input.

**Setup B — Diverse Dynamic Few‑Shot (MMR) + Guarded Retry Chain**
- Select examples using **Max Marginal Relevance** (relevant + diverse).
- Add a small **format guard**: if output is invalid, retry with a stricter prompt.

We run **two tasks**:
- Sentiment classification (positive/negative/neutral)
- Topic classification (sports/politics/technology/health/finance)

We compare:
- Fixed few-shot (baseline)
- Manual semantic selection (baseline)
- LangChain semantic selector (Setup A)
- LangChain MMR selector + guarded retry (Setup B)

Model: SmolLM2 via `transformers` pipeline (no API keys).


In [ ]:
# ============================================================
# Cell 1: Install dependencies
# ============================================================
# If you hit: ImportError cannot import name '_center' from numpy._core.umath
# do this, then Runtime -> Restart runtime, then continue.

%pip -q install --upgrade --no-cache-dir --force-reinstall "numpy==1.26.4"
%pip -q install --upgrade --no-cache-dir --force-reinstall "scipy==1.12.0" "scikit-learn==1.4.1.post1"
%pip -q install --upgrade --no-cache-dir transformers accelerate langchain langchain-core langchain-community \
  sentence-transformers faiss-cpu pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 174.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 134.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
dask-cudf-cu12 25.10.0 requires pandas<2.4.0dev0,>=2.0, but you have pandas 3.0.0 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatibl

In [ ]:
import numpy as np, scipy, sklearn
print("numpy:", np.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)

numpy: 1.26.4
scipy: 1.12.0
sklearn: 1.4.1.post1


## 2) Imports + configuration

In [ ]:
import os, re, json, time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

from langchain_community.llms import HuggingFacePipeline
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate
from langchain_core.example_selectors import (
    SemanticSimilarityExampleSelector,
    MaxMarginalRelevanceExampleSelector
)
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser

str_parser = StrOutputParser()

# Choose model size (fast -> better quality)
SMOLLM2_MODELS = {
    "135M": "HuggingFaceTB/SmolLM2-135M-Instruct",
    "360M": "HuggingFaceTB/SmolLM2-360M-Instruct",
    "1.7B": "HuggingFaceTB/SmolLM2-1.7B-Instruct",
}

MODEL_KEY = os.getenv("SMOLLM2_SIZE", "360M")
MODEL_ID = SMOLLM2_MODELS.get(MODEL_KEY, SMOLLM2_MODELS["360M"])

# Deterministic generation for evaluation
GEN_KWARGS = dict(
    max_new_tokens=64,
    do_sample=False,     # greedy
    temperature=0.0,
    top_p=1.0,
)

## 3) Load SmolLM2 locally (transformers pipeline → LangChain LLM)

We avoid API keys entirely. This runs on GPU if available.


In [ ]:
def load_hf_llm(model_id: str):
    use_cuda = torch.cuda.is_available()
    dtype = torch.float16 if use_cuda else torch.float32

    tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=dtype,
        device_map="auto",
    )

    gen_pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        return_full_text=False,
    )

    # IMPORTANT: pass generation kwargs here to avoid LC defaults like max_length=20
    llm = HuggingFacePipeline(pipeline=gen_pipe, pipeline_kwargs=GEN_KWARGS)
    return llm, tokenizer, model

llm, tokenizer, _ = load_hf_llm(MODEL_ID)
print("Loaded model:", MODEL_ID)
print("CUDA available:", torch.cuda.is_available())

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Loaded model: HuggingFaceTB/SmolLM2-360M-Instruct
CUDA available: True


/tmp/ipython-input-169295626.py:20: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=gen_pipe, pipeline_kwargs=GEN_KWARGS)


## 4) Datasets: exemplars bank + evaluation queries

We keep it small and readable. You can scale later.


In [ ]:
TOPICS = ["sports", "politics", "technology", "health", "finance"]

# ----------------------------
# Few-shot exemplar banks
# ----------------------------
SENTIMENT_EXAMPLES = [
    {"text": "Loved it. Smooth experience and fast support.", "label": "positive"},
    {"text": "This is the worst update. Everything is broken.", "label": "negative"},
    {"text": "It works as expected. Nothing special.", "label": "neutral"},
    {"text": "Great quality and quick delivery.", "label": "positive"},
    {"text": "Unacceptable. Crashes repeatedly.", "label": "negative"},
    {"text": "It is okay, not too impressive.", "label": "neutral"},
    {"text": "Fantastic results; exceeded expectations.", "label": "positive"},
    {"text": "Terrible performance and bad UX.", "label": "negative"},
]

TOPIC_EXAMPLES = [
    {"text": "The striker scored a hat-trick in the derby.", "label": "sports"},
    {"text": "Parliament debated the new tax bill today.", "label": "politics"},
    {"text": "A new GPU architecture improves training speed.", "label": "technology"},
    {"text": "Doctors recommend regular exercise for heart health.", "label": "health"},
    {"text": "Stocks fell after the earnings report missed estimates.", "label": "finance"},
    {"text": "The team advanced to the finals after a 3-1 win.", "label": "sports"},
    {"text": "The government announced new election guidelines.", "label": "politics"},
    {"text": "A major software update patches critical security flaws.", "label": "technology"},
    {"text": "A clinical trial shows improved outcomes for patients.", "label": "health"},
    {"text": "Inflation data affected bond yields and markets.", "label": "finance"},
]

# ----------------------------
# Evaluation sets (queries)
# ----------------------------
SENTIMENT_EVAL = [
    ("Customer service was helpful and quick.", "positive"),
    ("This is unacceptable and keeps crashing.", "negative"),
    ("Not bad, but not amazing either.", "neutral"),
    ("I’m very happy with the purchase.", "positive"),
    ("The update ruined everything.", "negative"),
    ("Works fine; nothing special.", "neutral"),
]

TOPIC_EVAL = [
    ("The team won 3-1 and advanced to the finals.", "sports"),
    ("The minister answered questions in parliament.", "politics"),
    ("The new transformer model reduced latency.", "technology"),
    ("Doctors advise better sleep habits.", "health"),
    ("Markets rallied after strong earnings.", "finance"),
    ("The striker was transferred for a record fee.", "sports"),
]

## 5) Utility: label normalization + metrics

In [ ]:
def normalize_label(x: str) -> str:
    if x is None:
        return ""
    x = x.strip().lower()
    # many small LMs add extra words; take first token
    x = re.split(r"[\s\n\t\.,;:]+", x)[0]
    return x

def accuracy(preds, golds):
    return float(np.mean([p == g for p, g in zip(preds, golds)])) if golds else 0.0

# Baseline 1 — Fixed few-shot (manual)

Classic approach: always use the same exemplars.


In [ ]:
# Fixed few-shot prompt for sentiment
sent_example_prompt = PromptTemplate.from_template("Text: {text}\nSentiment: {label}")
sent_fixed_prompt = FewShotPromptTemplate(
    examples=SENTIMENT_EXAMPLES[:4],          # fixed subset
    example_prompt=sent_example_prompt,
    prefix="Classify sentiment. Labels: positive, negative, neutral. Output ONLY the label.\nExamples:",
    suffix="Text: {text}\nSentiment:",
    input_variables=["text"],
)

# Fixed few-shot prompt for topic
topic_example_prompt = PromptTemplate.from_template("Text: {text}\nTopic: {label}")
topic_fixed_prompt = FewShotPromptTemplate(
    examples=TOPIC_EXAMPLES[:5],              # fixed subset
    example_prompt=topic_example_prompt,
    prefix=f"Classify topic. Labels: {', '.join(TOPICS)}. Output ONLY the label.\nExamples:",
    suffix="Text: {text}\nTopic:",
    input_variables=["text"],
)

sent_fixed_chain = sent_fixed_prompt | llm | str_parser
topic_fixed_chain = topic_fixed_prompt | llm | str_parser

def eval_fixed():
    # Sentiment
    s_preds, s_golds = [], []
    for text, gold in SENTIMENT_EVAL:
        pred = normalize_label(sent_fixed_chain.invoke({"text": text}))
        s_preds.append(pred); s_golds.append(gold)
    # Topic
    t_preds, t_golds = [], []
    for text, gold in TOPIC_EVAL:
        pred = normalize_label(topic_fixed_chain.invoke({"text": text}))
        t_preds.append(pred); t_golds.append(gold)

    return {
        "sentiment_acc": accuracy(s_preds, s_golds),
        "topic_acc": accuracy(t_preds, t_golds),
    }

eval_fixed()

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'top_p', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_to

{'sentiment_acc': 0.8333333333333334, 'topic_acc': 1.0}

# Baseline 2 — Manual semantic selection (normal Python)

Manual pipeline:
- embed exemplars
- embed query
- pick top-k
- format prompt by hand


In [ ]:
from sentence_transformers import SentenceTransformer

# Small, fast embedding model (open-source)
EMB_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
st = SentenceTransformer(EMB_MODEL)

def embed_texts(texts):
    return st.encode(texts, normalize_embeddings=True)

def topk_examples(query_text, examples, k=4):
    ex_texts = [e["text"] for e in examples]
    ex_emb = embed_texts(ex_texts)
    q_emb = embed_texts([query_text])[0]
    sims = (ex_emb @ q_emb)  # cosine sim because normalized
    idx = np.argsort(-sims)[:k]
    return [examples[i] for i in idx]

def manual_prompt(task, selected_examples, query_text):
    if task == "sentiment":
        header = "Classify sentiment. Labels: positive, negative, neutral. Output ONLY the label."
        lines = [header, "Examples:"]
        for ex in selected_examples:
            lines.append(f"Text: {ex['text']}\nSentiment: {ex['label']}")
        lines.append(f"Text: {query_text}\nSentiment:")
        return "\n\n".join(lines)

    if task == "topic":
        header = f"Classify topic. Labels: {', '.join(TOPICS)}. Output ONLY the label."
        lines = [header, "Examples:"]
        for ex in selected_examples:
            lines.append(f"Text: {ex['text']}\nTopic: {ex['label']}")
        lines.append(f"Text: {query_text}\nTopic:")
        return "\n\n".join(lines)

    raise ValueError("Unknown task")

def manual_run(task, query_text, k=4):
    bank = SENTIMENT_EXAMPLES if task=="sentiment" else TOPIC_EXAMPLES
    selected = topk_examples(query_text, bank, k=k)
    prompt = manual_prompt(task, selected, query_text)
    return llm.invoke(prompt)

def eval_manual_semantic():
    s_preds, s_golds = [], []
    for text, gold in SENTIMENT_EVAL:
        pred = normalize_label(manual_run("sentiment", text, k=4))
        s_preds.append(pred); s_golds.append(gold)

    t_preds, t_golds = [], []
    for text, gold in TOPIC_EVAL:
        pred = normalize_label(manual_run("topic", text, k=4))
        t_preds.append(pred); t_golds.append(gold)

    return {
        "sentiment_acc": accuracy(s_preds, s_golds),
        "topic_acc": accuracy(t_preds, t_golds),
    }

eval_manual_semantic()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

{'sentiment_acc': 0.8333333333333334, 'topic_acc': 1.0}

# Setup A — LangChain SemanticSimilarityExampleSelector (dynamic few-shot)

Same idea as manual top-k, but cleaner and reusable.


In [ ]:
# ============================================================
# Dynamic Few-Shot with LangChain
# ============================================================

# Install updated embeddings package (if not already installed)
!pip -q install -U langchain-huggingface

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_core.output_parsers import StrOutputParser

str_parser = StrOutputParser()

# ------------------------------------------------------------
# 1) Embedding model
# ------------------------------------------------------------
hf_emb = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# ------------------------------------------------------------
# 2) Build FAISS stores (IMPORTANT: store text + label in metadata)
# ------------------------------------------------------------
sent_store = FAISS.from_texts(
    texts=[e["text"] for e in SENTIMENT_EXAMPLES],
    embedding=hf_emb,
    metadatas=[
        {"text": e["text"], "label": e["label"]}
        for e in SENTIMENT_EXAMPLES
    ],
)

topic_store = FAISS.from_texts(
    texts=[e["text"] for e in TOPIC_EXAMPLES],
    embedding=hf_emb,
    metadatas=[
        {"text": e["text"], "label": e["label"]}
        for e in TOPIC_EXAMPLES
    ],
)

# ------------------------------------------------------------
# 3) Semantic Similarity Example Selectors
# ------------------------------------------------------------
sent_selector = SemanticSimilarityExampleSelector(
    vectorstore=sent_store,
    k=4,
)

topic_selector = SemanticSimilarityExampleSelector(
    vectorstore=topic_store,
    k=5,
)

# ------------------------------------------------------------
# 4) FewShotPromptTemplates using selector
# ------------------------------------------------------------
sent_dyn_prompt = FewShotPromptTemplate(
    example_selector=sent_selector,
    example_prompt=PromptTemplate.from_template(
        "Text: {text}\nSentiment: {label}"
    ),
    prefix=(
        "Classify sentiment. "
        "Labels: positive, negative, neutral. "
        "Output ONLY the label.\nExamples:"
    ),
    suffix="Text: {text}\nSentiment:",
    input_variables=["text"],
)

topic_dyn_prompt = FewShotPromptTemplate(
    example_selector=topic_selector,
    example_prompt=PromptTemplate.from_template(
        "Text: {text}\nTopic: {label}"
    ),
    prefix=(
        f"Classify topic. Labels: {', '.join(TOPICS)}. "
        "Output ONLY the label.\nExamples:"
    ),
    suffix="Text: {text}\nTopic:",
    input_variables=["text"],
)

# ------------------------------------------------------------
# 5) Chains
# ------------------------------------------------------------
sent_dyn_chain = sent_dyn_prompt | llm | str_parser
topic_dyn_chain = topic_dyn_prompt | llm | str_parser

# ------------------------------------------------------------
# 6) Debug: inspect selected examples (IMPORTANT sanity check)
# ------------------------------------------------------------
print("Selected examples for a negative sentiment query:")
print(sent_selector.select_examples({"text": "This product is terrible."}))

# ------------------------------------------------------------
# 7) Evaluation function
# ------------------------------------------------------------
def eval_langchain_semantic():
    s_preds, s_golds = [], []
    for text, gold in SENTIMENT_EVAL:
        pred = normalize_label(sent_dyn_chain.invoke({"text": text}))
        s_preds.append(pred)
        s_golds.append(gold)

    t_preds, t_golds = [], []
    for text, gold in TOPIC_EVAL:
        pred = normalize_label(topic_dyn_chain.invoke({"text": text}))
        t_preds.append(pred)
        t_golds.append(gold)

    return {
        "sentiment_acc": accuracy(s_preds, s_golds),
        "topic_acc": accuracy(t_preds, t_golds),
    }

# Run evaluation
eval_langchain_semantic()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.1.0 requires huggingface-hub<2.0,>=1.3.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.0 which is incompatible.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Selected examples for a negative sentiment query:
[{'text': 'Terrible performance and bad UX.', 'label': 'negative'}, {'text': 'Unacceptable. Crashes repeatedly.', 'label': 'negative'}, {'text': 'It works as expected. Nothing special.', 'label': 'neutral'}, {'text': 'This is the worst update. Everything is broken.', 'label': 'negative'}]


Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

{'sentiment_acc': 0.8333333333333334, 'topic_acc': 1.0}

# Setup B — LangChain MMR (Maximal Marginal Relevance) selector + Guarded Retry Chain

MMR chooses examples that are relevant *and* diverse.
Then we add a simple output-format guard and retry.


In [ ]:
# 1) MMR selectors
sent_mmr_selector = MaxMarginalRelevanceExampleSelector(
    vectorstore=sent_store,
    k=4,
    fetch_k=8,
)
topic_mmr_selector = MaxMarginalRelevanceExampleSelector(
    vectorstore=topic_store,
    k=5,
    fetch_k=10,
)

sent_mmr_prompt = FewShotPromptTemplate(
    example_selector=sent_mmr_selector,
    example_prompt=PromptTemplate.from_template("Text: {text}\nSentiment: {label}"),
    prefix="Classify sentiment. Output ONLY one token from: positive, negative, neutral.\nExamples:",
    suffix="Text: {text}\nSentiment:",
    input_variables=["text"],
)

topic_mmr_prompt = FewShotPromptTemplate(
    example_selector=topic_mmr_selector,
    example_prompt=PromptTemplate.from_template("Text: {text}\nTopic: {label}"),
    prefix=f"Classify topic. Output ONLY one token from: {', '.join(TOPICS)}.\nExamples:",
    suffix="Text: {text}\nTopic:",
    input_variables=["text"],
)

# 2) Guard + retry (simple chaining)
VALID_SENT = {"positive","negative","neutral"}
VALID_TOP  = set(TOPICS)

def is_valid_label(task, out_text):
    lab = normalize_label(out_text)
    if task == "sentiment":
        return lab in VALID_SENT
    if task == "topic":
        return lab in VALID_TOP
    return False

def strict_retry_prompt(task, text):
    if task == "sentiment":
        return (
            "Output exactly one word: positive or negative or neutral. "
            "No punctuation. No explanation.\n\n"
            f"TEXT:\n{text}\nLABEL:"
        )
    if task == "topic":
        return (
            f"Output exactly one word from: {', '.join(TOPICS)}. "
            "No punctuation. No explanation.\n\n"
            f"TEXT:\n{text}\nLABEL:"
        )
    raise ValueError("Unknown task")

def guarded_call(task, prompt_obj, inputs):
    out = (prompt_obj | llm | str_parser).invoke(inputs)
    if is_valid_label(task, out):
        return out
    return llm.invoke(strict_retry_prompt(task, inputs["text"]))

sent_guarded_chain = RunnableLambda(lambda x: guarded_call("sentiment", sent_mmr_prompt, x))
topic_guarded_chain = RunnableLambda(lambda x: guarded_call("topic", topic_mmr_prompt, x))

def eval_langchain_mmr_guarded():
    s_preds, s_golds = [], []
    for text, gold in SENTIMENT_EVAL:
        pred = normalize_label(sent_guarded_chain.invoke({"text": text}))
        s_preds.append(pred); s_golds.append(gold)

    t_preds, t_golds = [], []
    for text, gold in TOPIC_EVAL:
        pred = normalize_label(topic_guarded_chain.invoke({"text": text}))
        t_preds.append(pred); t_golds.append(gold)

    return {
        "sentiment_acc": accuracy(s_preds, s_golds),
        "topic_acc": accuracy(t_preds, t_golds),
    }

eval_langchain_mmr_guarded()

Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

{'sentiment_acc': 0.8333333333333334, 'topic_acc': 1.0}

# 10) Benchmark: compare all methods + timing

We measure accuracy and wall-clock time.


In [ ]:
def timed_eval(fn, name):
    t0 = time.perf_counter()
    out = fn()
    t1 = time.perf_counter()
    out["seconds"] = round(t1 - t0, 4)
    out["method"] = name
    return out

results = []
results.append(timed_eval(eval_fixed, "Fixed few-shot (manual)"))
results.append(timed_eval(eval_manual_semantic, "Manual semantic top-k (Python)"))
results.append(timed_eval(eval_langchain_semantic, "LangChain semantic selector (Setup A)"))
results.append(timed_eval(eval_langchain_mmr_guarded, "LangChain MMR + guard (Setup B)"))

df = pd.DataFrame(results)[["method","sentiment_acc","topic_acc","seconds"]]
df

Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `ma

,method,sentiment_acc,topic_acc,seconds
0,Fixed few-shot (manual),0.833333,1.0,32.3319
1,Manual semantic top-k (Python),0.833333,1.0,41.9853
2,LangChain semantic selector (Setup A),0.833333,1.0,34.9611
3,LangChain MMR + guard (Setup B),0.833333,1.0,32.7607


# 11) Why LangChain helps (practical view)

### Manual approach
- embeddings, top‑k, prompt assembly, retries → lots of glue code

### LangChain approach
- selectors are modular (`SemanticSimilarityExampleSelector` ↔ `MaxMarginalRelevanceExampleSelector`)
- prompts stay declarative (`FewShotPromptTemplate`)
- chaining stays readable (selector → prompt → model → guard → retry)

As you add tasks and prompt versions, this modularity becomes the main win.


## Optional exercises

1) Increase `k` and see if accuracy changes.
2) Add a third task and reuse the same selection machinery.
3) Switch `SMOLLM2_SIZE=1.7B` and compare quality vs speed.
